# Oye Mia — openWakeWord personalizado (español)
**Antes de ejecutar:** Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU**.
Ejecuta las celdas en orden (Mayús+Intro). Tiempo estimado en T4: 1,5 a 3 h.
Resultado: `oye_mia.onnx` con entrada `[1,16,96]`, salida `[1,1]`, opset 13, compatible con los `melspectrogram.onnx` y `embedding_model.onnx` originales.
Método: el del cuaderno oficial `automatic_model_training.ipynb` de dscripka/openWakeWord, con los arreglos de dependencias de 2026 (Python 3.12, `piper-phonemize`, torchaudio 2.x) que recoge alfiedennen/openwakeword-colab-2026. La diferencia clave: aquí los positivos se generan **directamente con Piper en español**, no con piper-sample-generator/LibriTTS en inglés.

In [22]:
import os, glob
def n(d):
    return len(glob.glob(d + '/*.wav'))
print('sesion viva, archivos:')
print('out:', sorted(os.listdir('/content/oye_mia/out')) if os.path.exists('/content/oye_mia/out') else 'NO EXISTE')
print('pos_train', n('/content/oye_mia/clips_pos_train'), 'pos_test', n('/content/oye_mia/clips_pos_test'), 'neg_train', n('/content/oye_mia/clips_neg_train'), 'neg_test', n('/content/oye_mia/clips_neg_test'))
print('Xtr en memoria:', 'Xtr' in dir())

sesion viva, archivos:
out: ['best.pt', 'fondo_es.wav', 'oye_mia.onnx']
pos_train 5000 pos_test 500 neg_train 5000 neg_test 500
Xtr en memoria: True


In [23]:
import torch, numpy as np, os
import torch.nn as nn
from pathlib import Path

class FCNBlock(nn.Module):
    def __init__(self, layer_dim):
        super().__init__()
        self.fcn_layer = nn.Linear(layer_dim, layer_dim)
        self.relu = nn.ReLU()
        self.layer_norm = nn.LayerNorm(layer_dim)
    def forward(self, x):
        return self.relu(self.layer_norm(self.fcn_layer(x)))

class Net(nn.Module):
    def __init__(self, input_shape=(16, 96), layer_dim=128, n_blocks=1, n_classes=1):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(input_shape[0] * input_shape[1], layer_dim)
        self.relu1 = nn.ReLU()
        self.layernorm1 = nn.LayerNorm(layer_dim)
        self.blocks = nn.ModuleList([FCNBlock(layer_dim) for _ in range(n_blocks)])
        self.last_layer = nn.Linear(layer_dim, n_classes)
        self.last_act = nn.Sigmoid()
    def forward(self, x):
        x = self.relu1(self.layernorm1(self.layer1(self.flatten(x))))
        for block in self.blocks:
            x = block(x)
        return self.last_act(self.last_layer(x))

OUT = Path('/content/oye_mia/out')
OUT.mkdir(parents=True, exist_ok=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(2024)
net = Net().to(dev)
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
Xtr_t = torch.tensor(Xtr); ytr_t = torch.tensor(ytr)[:, None]
Xte_t = torch.tensor(Xte, device=dev); yte_t = torch.tensor(yte, device=dev)[:, None]
npos = max(1, (yte_t == 1).sum().item()); nneg = max(1, (yte_t == 0).sum().item())
STEPS = 50000
best = None; fallback = None
for step in range(1, STEPS + 1):
    net.train()
    idx = torch.randint(0, len(Xtr_t), (512,))
    xb, yb = Xtr_t[idx].to(dev), ytr_t[idx].to(dev)
    neg_w = 1 + 999 * (step / STEPS)
    w = torch.where(yb == 0, torch.tensor(neg_w, device=dev), torch.tensor(1.0, device=dev))
    pred = net(xb)
    loss = (torch.nn.functional.binary_cross_entropy(pred, yb, reduction='none') * w).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        net.eval()
        with torch.no_grad():
            pv = net(Xte_t)
        for th in (0.5, 0.6, 0.7, 0.8):
            rec = ((pv > th) & (yte_t == 1)).sum().item() / npos
            fp = int(((pv > th) & (yte_t == 0)).sum().item())
            if fp <= 10 and (best is None or rec > best[0]):
                best = (rec, fp, th, step, {k: v.cpu().clone() for k, v in net.state_dict().items()})
                print(f'  >> mejor: paso {step} umbral {th} recall {rec:.3f} FP {fp}')
            score = rec - 2.0 * (fp / nneg)
            if fallback is None or score > fallback[0]:
                fallback = (score, rec, fp, th, step, {k: v.cpu().clone() for k, v in net.state_dict().items()})
        if step % 5000 == 0:
            rec5 = ((pv > 0.5) & (yte_t == 1)).sum().item() / npos
            fp5 = int(((pv > 0.5) & (yte_t == 0)).sum().item())
            print(f'paso {step} loss {loss.item():.4f} recall@0.5 {rec5:.3f} FP@0.5 {fp5}')
if best is not None:
    rec, fp, th, step, state = best
    print(f'ELEGIDO: paso {step} umbral {th} recall {rec:.3f} FP {fp}')
else:
    score, rec, fp, th, step, state = fallback
    print(f'SIN checkpoint con FP<=10; respaldo: paso {step} umbral {th} recall {rec:.3f} FP {fp}')
torch.save(state, OUT / 'best_v2.pt')
print('best_v2.pt guardado:', (OUT / 'best_v2.pt').exists(), os.path.getsize(OUT / 'best_v2.pt'))
import onnx
net2 = Net(); net2.load_state_dict(state); net2 = net2.cpu().eval()
out_path = str(OUT / 'oye_mia_v2.onnx')
torch.onnx.export(net2, torch.rand(1, 16, 96), out_path, opset_version=13, input_names=['input'], output_names=['output'], dynamic_axes=None, dynamo=False)
chk = onnx.load(out_path); onnx.checker.check_model(chk)
def shp(t):
    return [d.dim_value for d in t.type.tensor_type.shape.dim]
print('opset:', chk.opset_import[0].version)
print('entrada:', shp(chk.graph.input[0]), '| salida:', shp(chk.graph.output[0]))
print('tamano onnx KB:', os.path.getsize(out_path) / 1024)
from google.colab import files
files.download(out_path)
print('DESCARGA DISPARADA')

  >> mejor: paso 3000 umbral 0.6 recall 0.546 FP 10
paso 5000 loss 0.0219 recall@0.5 0.860 FP@0.5 54
paso 10000 loss 0.0008 recall@0.5 0.888 FP@0.5 62
paso 15000 loss 0.4503 recall@0.5 0.772 FP@0.5 23
paso 20000 loss 0.3063 recall@0.5 0.776 FP@0.5 32
  >> mejor: paso 23500 umbral 0.8 recall 0.550 FP 9
paso 25000 loss 2.1241 recall@0.5 0.220 FP@0.5 1
paso 30000 loss 0.1173 recall@0.5 0.830 FP@0.5 49
paso 35000 loss 15.1012 recall@0.5 0.000 FP@0.5 0
  >> mejor: paso 36500 umbral 0.7 recall 0.570 FP 9
paso 40000 loss 27.5189 recall@0.5 0.000 FP@0.5 0
paso 45000 loss 25.8627 recall@0.5 0.000 FP@0.5 0
paso 50000 loss 27.2193 recall@0.5 0.000 FP@0.5 0
ELEGIDO: paso 36500 umbral 0.7 recall 0.570 FP 9
best_v2.pt guardado: True 858949
opset: 13
entrada: [1, 16, 96] | salida: [1, 1]
tamano onnx KB: 839.2412109375


/tmp/ipykernel_1223/2574773359.py:78: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(net2, torch.rand(1, 16, 96), out_path, opset_version=13, input_names=['input'], output_names=['output'], dynamic_axes=None, dynamo=False)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DESCARGA DISPARADA


In [ ]:
import torch, sys
print("Python", sys.version)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "SIN GPU - cambia a T4 antes de seguir")
assert torch.cuda.is_available(), "Activa la GPU T4 y reinicia el entorno"

Python 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
GPU: Tesla T4


## 1. Instalación
Instala versiones que conviven en Colab 2026: `piper-tts` sin deps al final (para que no rompa torch), `openwakeword` desde el repo oficial, y ONNX con opset 13.

In [ ]:
!apt-get -qq update && apt-get -qq install -y espeak-ng ffmpeg sox
!pip install -q "numpy<2.3" scipy soundfile librosa onnx onnxruntime tqdm pyyaml requests torchinfo torchmetrics
!pip install -q --no-deps git+https://github.com/dscripka/openWakeWord.git
!wget -q https://github.com/rhasspy/piper/releases/download/v1.2.0/piper_amd64.tar.gz -O /content/piper.tar.gz
!tar -xzf /content/piper.tar.gz -C /content/
import openwakeword, onnxruntime, subprocess
_r = subprocess.run(['/content/piper/piper','--help'], capture_output=True, text=True)
print('piper binario responde, codigo', _r.returncode)
print('openwakeword OK, onnxruntime', onnxruntime.__version__)
from openwakeword.utils import download_models
download_models()
print('modelos del extractor descargados')


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../00-libpcaudio0_1.2-2build3_amd64.deb ...
Unpacking libpcaudio0:amd64 (1.2-2build3) ...
Selecting previously unselected package libsonic0:amd64.
Preparing to unpack .../01-libsonic0_0.2.0-13build1_amd64.deb ...
Unpacking libsonic0:amd64 (0.2.0-13build1) ...
Selecting previously unselected package espeak-ng-data:amd64.
Preparing to unpack .../02-espeak-ng-data_1.51+dfsg-12build1_amd64.deb ...
Unpacking espeak-ng-data:amd64 (1.51+dfsg-12build1) ...
Selecting previously unselected package libespeak-ng1:amd64.
Preparing to unpack .../03-libespeak-ng1_1.51+dfsg-12build1_amd64.deb ...
Unpacking libespeak-ng1:amd64 (1.51+dfsg-12build1) ...
Selecting previ

embedding_model.tflite: 100%|██████████| 1.33M/1.33M [00:00<00:00, 43.4MiB/s]
embedding_model.onnx: 100%|██████████| 1.33M/1.33M [00:00<00:00, 36.1MiB/s]
melspectrogram.tflite: 100%|██████████| 1.09M/1.09M [00:00<00:00, 39.4MiB/s]
melspectrogram.onnx: 100%|██████████| 1.09M/1.09M [00:00<00:00, 32.6MiB/s]
silero_vad.onnx: 100%|██████████| 1.81M/1.81M [00:00<00:00, 55.4MiB/s]
alexa_v0.1.tflite: 100%|██████████| 855k/855k [00:00<00:00, 24.6MiB/s]
alexa_v0.1.onnx: 100%|██████████| 854k/854k [00:00<00:00, 31.2MiB/s]
hey_mycroft_v0.1.tflite: 100%|██████████| 860k/860k [00:00<00:00, 30.3MiB/s]
hey_mycroft_v0.1.onnx: 100%|██████████| 858k/858k [00:00<00:00, 32.3MiB/s]
hey_jarvis_v0.1.tflite: 100%|██████████| 1.28M/1.28M [00:00<00:00, 36.5MiB/s]
hey_jarvis_v0.1.onnx: 100%|██████████| 1.27M/1.27M [00:00<00:00, 34.1MiB/s]
hey_rhasspy_v0.1.tflite: 100%|██████████| 416k/416k [00:00<00:00, 16.7MiB/s]
hey_rhasspy_v0.1.onnx: 100%|██████████| 204k/204k [00:00<00:00, 10.4MiB/s]
timer_v0.1.tflite: 100%|█

modelos del extractor descargados


## 2. Configuración
Todo lo que puedes tocar está aquí: frase, voces, cantidades y pasos (30 000 por defecto, dentro del rango oficial 20 000-50 000).

In [ ]:
from pathlib import Path
BASE = Path('/content/oye_mia')
for d in ['voices','clips_pos_train','clips_pos_test','clips_neg_train','clips_neg_test','features','out']:
    (BASE/d).mkdir(parents=True, exist_ok=True)
TARGET_VARIANTS = ['oye mía','oye, mía','oye mia','¡oye mía!','oye mía?']
ADVERSARIAL = ['oye mira','hoy mía','oye tía','oye María','oye mamá','mía','oye','mira','o sea mía','voy a ir','lo mío','joya mía']
GENERIC_NEG = ['buenos días','cuánto cuesta','pon la música','qué hora es','llama a mamá','abre la puerta','hace mucho calor hoy','vamos a la tienda','los niños ya comieron','mañana salimos temprano','apaga la luz','sube el volumen']
VOICES = {
    'es_MX-claude-high':'es/es_MX/claude/high/es_MX-claude-high',
    'es_MX-ald-medium':'es/es_MX/ald/medium/es_MX-ald-medium',
    'es_AR-daniela-high':'es/es_AR/daniela/high/es_AR-daniela-high',
    'es_ES-davefx-medium':'es/es_ES/davefx/medium/es_ES-davefx-medium',
    'es_ES-sharvard-medium':'es/es_ES/sharvard/medium/es_ES-sharvard-medium',
    'es_ES-carlfm-x_low':'es/es_ES/carlfm/x_low/es_ES-carlfm-x_low',
    'es_ES-mls_9972-low':'es/es_ES/mls_9972/low/es_ES-mls_9972-low',
    'es_ES-mls_10246-low':'es/es_ES/mls_10246/low/es_ES-mls_10246-low',
}
N_POS_TRAIN, N_POS_TEST = 5000, 500
N_NEG_TRAIN, N_NEG_TEST = 5000, 500
TRAIN_STEPS = 50000
SEED = 1234
print('Config OK')


Config OK


In [ ]:
# Respaldo en Drive: si el montaje funciona, cada etapa cara se copia a Drive
# y una sesion reciclada ya no lo borra todo. Si falla, se sigue en local.
DRIVE_OK = False
BACKUP = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    from pathlib import Path as _P
    BACKUP = _P('/content/drive/MyDrive/oye_mia_backup')
    BACKUP.mkdir(parents=True, exist_ok=True)
    DRIVE_OK = True
    print('Drive montado, respaldo en', BACKUP)
except Exception as e:
    print('Drive no montado, se sigue en local:', type(e).__name__)


Drive no montado, se sigue en local: ValueError


## 3. Descarga de voces Piper en español
Descarga `.onnx` + `.onnx.json` de cada voz desde huggingface.co/rhasspy/piper-voices. La estructura de URL `.../resolve/v1.0.0/es/...` es la documentada para ese repo.

In [ ]:
import requests
for name, rel in VOICES.items():
    for ext in ['.onnx', '.onnx.json']:
        dest = BASE/'voices'/(name+ext)
        if dest.exists() and dest.stat().st_size>1000:
            continue
        url = f'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/{rel}{ext}'
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        dest.write_bytes(r.content)
        print(name+ext, len(r.content)//1024, 'KB')
print('Voces listas:', sorted(q.name for q in (BASE/'voices').glob('*.onnx')))


es_MX-claude-high.onnx 61642 KB
es_MX-claude-high.onnx.json 4 KB
es_MX-ald-medium.onnx 61720 KB
es_MX-ald-medium.onnx.json 4 KB
es_AR-daniela-high.onnx 111522 KB
es_AR-daniela-high.onnx.json 7 KB
es_ES-davefx-medium.onnx 61720 KB
es_ES-davefx-medium.onnx.json 4 KB
es_ES-sharvard-medium.onnx 74935 KB
es_ES-sharvard-medium.onnx.json 4 KB
es_ES-carlfm-x_low.onnx 27471 KB
es_ES-carlfm-x_low.onnx.json 4 KB
es_ES-mls_9972-low.onnx 61625 KB
es_ES-mls_9972-low.onnx.json 4 KB
es_ES-mls_10246-low.onnx 61625 KB
es_ES-mls_10246-low.onnx.json 4 KB
Voces listas: ['es_AR-daniela-high.onnx', 'es_ES-carlfm-x_low.onnx', 'es_ES-davefx-medium.onnx', 'es_ES-mls_10246-low.onnx', 'es_ES-mls_9972-low.onnx', 'es_ES-sharvard-medium.onnx', 'es_MX-ald-medium.onnx', 'es_MX-claude-high.onnx']


## 4. Generación de clips (positivos, adversarios y genéricos)
Usa la API actual de Piper (`PiperVoice` + `SynthesisConfig`) variando `length_scale` 0,8-1,3, `noise_scale` y `noise_w_scale`, y el hablante cuando la voz tiene 2 (sharvard). **Train y test usan semillas y combinaciones distintas**, así el test es de verdad no visto. Cada clip se normaliza a 16 kHz mono y 2 s (recorte/relleno con silencio), que es lo que espera openWakeWord.

In [ ]:
import random, subprocess, tempfile, os, numpy as np, soundfile as sf
from concurrent.futures import ThreadPoolExecutor
PIPER_BIN = '/content/piper/piper'
SR = 16000
def synth(voice_name, text, rng, holdout=False):
    n_spk = 2 if 'sharvard' in voice_name else 1
    fd, tmp = tempfile.mkstemp(suffix='.wav'); os.close(fd)
    cmd = [PIPER_BIN, '--model', str(BASE/'voices'/f'{voice_name}.onnx'), '--output_file', tmp,
        '--length-scale', str(rng.uniform(0.95,1.15) if holdout else rng.uniform(0.8,1.3)),
        '--noise-scale', str(rng.uniform(0.5,0.8)),
        '--noise-w-scale', str(rng.uniform(0.6,0.9))]
    if n_spk > 1:
        cmd += ['--speaker', str(rng.randrange(n_spk))]
    r = subprocess.run(cmd, input=text.encode(), capture_output=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.decode()[-500:])
    audio, src_sr = sf.read(tmp)
    os.remove(tmp)
    audio = audio.astype(np.float32)
    if audio.ndim > 1:
        audio = audio[:,0]
    if src_sr != SR:
        x_old = np.linspace(0,1,len(audio))
        x_new = np.linspace(0,1,int(len(audio)*SR/src_sr))
        audio = np.interp(x_new, x_old, audio).astype(np.float32)
    return audio
def to_2s(audio, rng):
    out = np.zeros(SR*2, dtype=np.float32)
    start = rng.randrange(0, max(1, SR*2-len(audio))) if len(audio)<SR*2 else 0
    seg = audio[:SR*2]
    out[start:start+len(seg)] = seg[:SR*2-start]
    peak = np.max(np.abs(out))+1e-9
    return (out/max(peak,1.0)*0.89)
def gen_set(texts_fn, total, folder, seed):
    names = list(VOICES)
    def one(i):
        rng = random.Random(seed*100000 + i)
        text = texts_fn(rng)
        vn = rng.choice(names)
        sf.write(BASE/folder/f'{i:05d}_{vn}.wav', to_2s(synth(vn, text, rng, holdout='test' in folder), rng), SR)
        if i%500==0:
            print(folder, i, '/', total)
    with ThreadPoolExecutor(max_workers=4) as ex:
        list(ex.map(one, range(total)))
gen_set(lambda r: r.choice(TARGET_VARIANTS), N_POS_TRAIN, 'clips_pos_train', SEED)
gen_set(lambda r: r.choice(TARGET_VARIANTS), N_POS_TEST, 'clips_pos_test', SEED+999)
gen_set(lambda r: r.choice(ADVERSARIAL if r.random()<0.5 else GENERIC_NEG), N_NEG_TRAIN, 'clips_neg_train', SEED+1)
gen_set(lambda r: r.choice(ADVERSARIAL if r.random()<0.5 else GENERIC_NEG), N_NEG_TEST, 'clips_neg_test', SEED+1000)
print('Clips generados')


clips_pos_train 0 / 5000
clips_pos_train 500 / 5000
clips_pos_train 1000 / 5000
clips_pos_train 1500 / 5000
clips_pos_train 2000 / 5000
clips_pos_train 2500 / 5000
clips_pos_train 3000 / 5000
clips_pos_train 3500 / 5000
clips_pos_train 4000 / 5000
clips_pos_train 4500 / 5000
clips_pos_test 0 / 500
clips_neg_train 0 / 5000
clips_neg_train 500 / 5000
clips_neg_train 1000 / 5000
clips_neg_train 1500 / 5000
clips_neg_train 2000 / 5000
clips_neg_train 2500 / 5000
clips_neg_train 3000 / 5000
clips_neg_train 3500 / 5000
clips_neg_train 4000 / 5000
clips_neg_train 4500 / 5000
clips_neg_test 0 / 500
Clips generados


In [ ]:
# Respaldo de clips a Drive (solo si el montaje funciono)
if DRIVE_OK:
    import shutil
    for d in ['clips_pos_train','clips_pos_test','clips_neg_train','clips_neg_test','voices']:
        shutil.copytree(BASE/d, BACKUP/d, dirs_exist_ok=True)
    print('Clips respaldados en Drive')
else:
    print('Sin Drive, respaldo omitido')


Sin Drive, respaldo omitido


## 5. Aumento de datos
Replica el aumento del cuaderno oficial sin depender de datasets externos que ya no descargan: convolución con una respuesta de impulso de sala sintética (ruido con decaimiento exponencial), ruido de fondo (ruido coloreado + zumbido de red + golpes suaves tipo casa), y ganancia aleatoria. Se aplica **solo al train**, nunca al test.

In [ ]:
import numpy as np, soundfile as sf
rng = np.random.default_rng(SEED)
def augment(x):
    if rng.random()<0.7:
        t = int(rng.integers(800, 6000))
        rir = rng.standard_normal(t)*np.exp(-np.linspace(0, rng.uniform(3,6), t))
        rir /= (np.max(np.abs(rir))+1e-9)
        y = np.convolve(x, rir)[:len(x)]
    else:
        y = x.copy()
    noise = np.cumsum(rng.standard_normal(len(y)))
    noise /= (np.max(np.abs(noise))+1e-9)
    noise += 0.05*np.sin(2*np.pi*60*np.arange(len(y))/16000)
    snr = 10**(rng.uniform(5,25)/20)
    y = y + noise/max(1e-9, np.max(np.abs(y)))*snr**-1*0.5
    y *= 10**(rng.uniform(-12,0)/20)
    return np.clip(y, -1, 1).astype(np.float32)
for folder in ['clips_pos_train','clips_neg_train']:
    for pth in sorted((BASE/folder).glob('*.wav')):
        x, sr = sf.read(pth)
        sf.write(pth, augment(x.astype(np.float32)), sr)
print('Aumento aplicado al train')


Aumento aplicado al train


## 6. Extracción de características (sin tocar el extractor estándar)
Usa `openwakeword.utils.AudioFeatures`, que por dentro ejecuta los `melspectrogram.onnx` y `embedding_model.onnx` originales. De cada clip de 2 s se toman **16 vectores de 96** (la ventana que ve el clasificador, igual que `alexa_v0.1.onnx`).

In [ ]:
import numpy as np, soundfile as sf
from openwakeword.utils import AudioFeatures
af = AudioFeatures()
def feats(folder):
    out = []
    for pth in sorted((BASE/folder).glob('*.wav')):
        x, _sr = sf.read(pth)
        emb = np.array(af._get_embeddings((x*32767).astype(np.int16)), dtype=np.float32)
        if len(emb) < 16:
            emb = np.pad(emb, ((0,16-len(emb)),(0,0)))
        out.append(emb[-16:])
    return np.stack(out)
Xpos_tr, Xneg_tr = feats('clips_pos_train'), feats('clips_neg_train')
Xpos_te, Xneg_te = feats('clips_pos_test'), feats('clips_neg_test')
Xtr = np.concatenate([Xpos_tr, Xneg_tr])
ytr = np.array([1]*len(Xpos_tr)+[0]*len(Xneg_tr), dtype=np.float32)
Xte = np.concatenate([Xpos_te, Xneg_te])
yte = np.array([1]*len(Xpos_te)+[0]*len(Xneg_te), dtype=np.float32)
np.savez(BASE/'features'/'data.npz', Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte)
print('Shapes:', Xtr.shape, Xte.shape, '| entrada:', Xtr.shape[1:])
assert Xtr.shape[1:] == (16,96)


Shapes: (10000, 16, 96) (1000, 16, 96) | entrada: (16, 96)


## 7. Entrenamiento
Usa la misma clase DNN oficial (`openwakeword.train.Model`, `layer_dim=128`) y un bucle equivalente a `auto_train`: peso creciente para los negativos (prioriza pocas falsas activaciones), validación periódica y guarda el mejor checkpoint por recall con FP bajo. Si prefieres el `auto_train` original, su firma y su secuencia están en el repo oficial.

In [ ]:
import torch, numpy as np
import torch.nn as nn
# Misma arquitectura DNN oficial de openwakeword.train.Model (layer_dim=128, 1 bloque),
# definida localmente para no importar openwakeword.train (que arrastra data.py y sus dependencias).
class FCNBlock(nn.Module):
    def __init__(self, layer_dim):
        super().__init__()
        self.fcn_layer = nn.Linear(layer_dim, layer_dim)
        self.relu = nn.ReLU()
        self.layer_norm = nn.LayerNorm(layer_dim)
    def forward(self, x):
        return self.relu(self.layer_norm(self.fcn_layer(x)))
class Net(nn.Module):
    def __init__(self, input_shape=(16,96), layer_dim=128, n_blocks=1, n_classes=1):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(input_shape[0]*input_shape[1], layer_dim)
        self.relu1 = nn.ReLU()
        self.layernorm1 = nn.LayerNorm(layer_dim)
        self.blocks = nn.ModuleList([FCNBlock(layer_dim) for _ in range(n_blocks)])
        self.last_layer = nn.Linear(layer_dim, n_classes)
        self.last_act = nn.Sigmoid()
    def forward(self, x):
        x = self.relu1(self.layernorm1(self.layer1(self.flatten(x))))
        for block in self.blocks:
            x = block(x)
        return self.last_act(self.last_layer(x))
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
net = Net().to(dev)
opt = torch.optim.Adam(net.parameters(), lr=5e-4)
Xtr_t = torch.tensor(Xtr); ytr_t = torch.tensor(ytr)[:,None]
Xte_t = torch.tensor(Xte, device=dev); yte_t = torch.tensor(yte, device=dev)[:,None]
best_recall = 0.0
for step in range(1, TRAIN_STEPS+1):
    net.train()
    idx = torch.randint(0, len(Xtr_t), (512,))
    xb, yb = Xtr_t[idx].to(dev), ytr_t[idx].to(dev)
    neg_w = 1 + 29*(step/TRAIN_STEPS)
    w = torch.where(yb==0, torch.tensor(neg_w, device=dev), torch.tensor(1.0, device=dev))
    pred = net(xb)
    loss = (torch.nn.functional.binary_cross_entropy(pred, yb, reduction='none')*w).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 1000 == 0:
        net.eval()
        with torch.no_grad():
            pv = net(Xte_t)
        rec = ((pv>0.5)&(yte_t==1)).sum().item()/max(1,(yte_t==1).sum().item())
        fp = ((pv>0.5)&(yte_t==0)).sum().item()
        print(f'paso {step} loss {loss.item():.4f} recall_test {rec:.3f} FP_test {fp}')
        if fp <= 5 and rec > best_recall:
            best_recall = rec
            torch.save({k: v.cpu().clone() for k,v in net.state_dict().items()}, BASE/'out'/'best.pt')
print('Mejor recall (con <=5 FP en test):', best_recall)


paso 1000 loss 0.0041 recall_test 0.866 FP_test 50
paso 2000 loss 0.0003 recall_test 0.866 FP_test 45
paso 3000 loss 0.0001 recall_test 0.864 FP_test 44
paso 4000 loss 0.0001 recall_test 0.866 FP_test 42
paso 5000 loss 0.0000 recall_test 0.866 FP_test 42
paso 6000 loss 0.0000 recall_test 0.866 FP_test 41
paso 7000 loss 0.0000 recall_test 0.866 FP_test 41
paso 8000 loss 0.0000 recall_test 0.866 FP_test 41
paso 9000 loss 0.0000 recall_test 0.864 FP_test 43
paso 10000 loss 0.0000 recall_test 0.864 FP_test 44
paso 11000 loss 0.0000 recall_test 0.864 FP_test 44
paso 12000 loss 0.0000 recall_test 0.864 FP_test 44
paso 13000 loss 0.0000 recall_test 0.866 FP_test 44
paso 14000 loss 0.0000 recall_test 0.866 FP_test 45
paso 15000 loss 0.0000 recall_test 0.866 FP_test 44
paso 16000 loss 0.0000 recall_test 0.868 FP_test 44
paso 17000 loss 0.0000 recall_test 0.868 FP_test 45
paso 18000 loss 0.0000 recall_test 0.868 FP_test 45
paso 19000 loss 0.0000 recall_test 0.866 FP_test 44
paso 20000 loss 0.000

## 8. Exportación a ONNX (opset 13, forma fija)
Exporta **sin ejes dinámicos**, para que la entrada quede exactamente `[1,16,96]` y la salida `[1,1]`, y lo comprueba con el paquete `onnx`.

In [ ]:
import torch, onnx
net.load_state_dict(torch.load(BASE/"out"/"best.pt")); net = net.to("cpu").eval()
out_path = str(BASE/"out"/"oye_mia.onnx")
torch.onnx.export(net, torch.rand(1,16,96), out_path, opset_version=13,
input_names=["input"], output_names=["output"], dynamic_axes=None, dynamo=False)
chk = onnx.load(out_path); onnx.checker.check_model(chk)
def shape(t): return [d.dim_value for d in t.type.tensor_type.shape.dim]
print("opset:", chk.opset_import[0].version)
print("entrada:", shape(chk.graph.input[0]), "| salida:", shape(chk.graph.output[0]))
assert shape(chk.graph.input[0])==[1,16,96] and shape(chk.graph.output[0])==[1,1]

FileNotFoundError: [Errno 2] No such file or directory: '/content/oye_mia/out/best.pt'

## 9. Validación real con openWakeWord (clips no vistos)
Carga el modelo **por la vía normal de inferencia** (`openwakeword.Model`, streaming en bloques de 1280 muestras) sobre los clips hold-out, barre umbrales, y estima falsas activaciones por hora con un audio largo en español sin la frase (generado con Piper uniendo frases genéricas; sustitúyelo por una grabación real tuya de 1 h si la tienes, será más fiable).

In [ ]:
import random, numpy as np, soundfile as sf
from openwakeword.model import Model
oww = Model(wakeword_models=[str(BASE/'out'/'oye_mia.onnx')], inference_framework='onnx')
def score_clip(pth):
    oww.reset()
    x, _sr = sf.read(pth)
    x = (x*32767).astype(np.int16)
    best = 0.0
    for i in range(0, len(x), 1280):
        s = oww.predict(x[i:i+1280])
        best = max(best, list(s.values())[0])
    return best
pos = [score_clip(pth) for pth in sorted((BASE/'clips_pos_test').glob('*.wav'))]
neg = [score_clip(pth) for pth in sorted((BASE/'clips_neg_test').glob('*.wav'))]
for th in [0.3,0.4,0.5,0.6,0.7]:
    print(f'umbral {th}: aciertos {np.mean(np.array(pos)>=th):.1%} | FP adversarios {np.mean(np.array(neg)>=th):.1%}')
rng2 = random.Random(7)
long_audio = []
vn0 = list(VOICES)[0]
for _j in range(300):
    long_audio.append(synth(vn0, rng2.choice(GENERIC_NEG), rng2, holdout=True))
la = np.concatenate(long_audio)
sf.write(BASE/'out'/'fondo_es.wav', la, 16000)
oww.reset()
la16 = (la*32767).astype(np.int16)
scores = []
for i in range(0, len(la16), 1280):
    scores.append(list(oww.predict(la16[i:i+1280]).values())[0])
horas = len(la)/16000/3600
for th in [0.5,0.6,0.7]:
    activ = sum(1 for a,b in zip(scores, scores[1:]) if a<th<=b)
    print(f'umbral {th}: falsas activaciones/hora (estimado, {horas:.2f} h) = {activ/max(horas,1e-9):.2f}')
print('UMBRAL RECOMENDADO: empieza en 0.5; sube a 0.6 si en tu parlante da >0.5 FP/hora')


umbral 0.3: aciertos 84.6% | FP adversarios 9.2%
umbral 0.4: aciertos 84.4% | FP adversarios 8.8%
umbral 0.5: aciertos 83.6% | FP adversarios 7.6%
umbral 0.6: aciertos 83.2% | FP adversarios 7.4%
umbral 0.7: aciertos 82.4% | FP adversarios 7.0%
umbral 0.5: falsas activaciones/hora (estimado, 0.12 h) = 112.78
umbral 0.6: falsas activaciones/hora (estimado, 0.12 h) = 104.10
umbral 0.7: falsas activaciones/hora (estimado, 0.12 h) = 104.10
UMBRAL RECOMENDADO: empieza en 0.5; sube a 0.6 si en tu parlante da >0.5 FP/hora


In [ ]:
import random, numpy as np, soundfile as sf
from openwakeword.model import Model
oww = Model(wakeword_models=[str(BASE/'out'/'oye_mia_v2.onnx')], inference_framework='onnx')
def score_clip(pth):
    oww.reset()
    x, _sr = sf.read(pth)
    x = (x*32767).astype(np.int16)
    best = 0.0
    for i in range(0, len(x), 1280):
        s = oww.predict(x[i:i+1280])
        best = max(best, list(s.values())[0])
    return best
pos = [score_clip(pth) for pth in sorted((BASE/'clips_pos_test').glob('*.wav'))]
neg = [score_clip(pth) for pth in sorted((BASE/'clips_neg_test').glob('*.wav'))]
for th in [0.3,0.4,0.5,0.6,0.7]:
    print(f'umbral {th}: aciertos {np.mean(np.array(pos)>=th):.1%} | FP adversarios {np.mean(np.array(neg)>=th):.1%}')
rng2 = random.Random(7)
long_audio = []
vn0 = list(VOICES)[0]
for _j in range(300):
    long_audio.append(synth(vn0, rng2.choice(GENERIC_NEG), rng2, holdout=True))
la = np.concatenate(long_audio)
sf.write(BASE/'out'/'fondo_es.wav', la, 16000)
oww.reset()
la16 = (la*32767).astype(np.int16)
scores = []
for i in range(0, len(la16), 1280):
    scores.append(list(oww.predict(la16[i:i+1280]).values())[0])
horas = len(la)/16000/3600
for th in [0.5,0.6,0.7]:
    activ = sum(1 for a, b in zip(scores, scores[1:]) if a < th <= b)
    print(f'umbral {th}: falsas activaciones/hora (estimado, {horas:.2f} h) = {activ/max(horas,1e-9):.2f}')
print('UMBRAL RECOMENDADO: empieza en 0.5; sube a 0.6 si en tu parlante da > 0.5 FP/hora')

umbral 0.3: aciertos 87.8% | FP adversarios 13.4%
umbral 0.4: aciertos 86.0% | FP adversarios 12.2%
umbral 0.5: aciertos 85.0% | FP adversarios 10.6%
umbral 0.6: aciertos 84.2% | FP adversarios 10.0%
umbral 0.7: aciertos 82.0% | FP adversarios 8.2%
umbral 0.5: falsas activaciones/hora (estimado, 0.12 h) = 104.07
umbral 0.6: falsas activaciones/hora (estimado, 0.12 h) = 69.38
umbral 0.7: falsas activaciones/hora (estimado, 0.12 h) = 52.04
UMBRAL RECOMENDADO: empieza en 0.5; sube a 0.6 si en tu parlante da > 0.5 FP/hora


## 10. Descarga
Descarga `oye_mia.onnx` a tu computadora. Úsalo junto a los `melspectrogram.onnx` y `embedding_model.onnx` que ya tiene tu parlante, con el umbral de la celda anterior.

In [ ]:
from google.colab import files
import shutil; shutil.copy(BASE/"out"/"oye_mia.onnx", "/content/oye_mia.onnx")
if DRIVE_OK:
    import shutil as _sh
    _sh.copy('/content/oye_mia.onnx', BACKUP/'oye_mia.onnx')
    print('Copiado a Drive:', BACKUP/'oye_mia.onnx')
files.download("/content/oye_mia.onnx")
print("Descargado: oye_mia.onnx")

FileNotFoundError: [Errno 2] No such file or directory: '/content/oye_mia/out/oye_mia.onnx'

In [ ]:
from pathlib import Path
import os
print('data.npz existe:', os.path.exists('/content/oye_mia/data.npz'))
print('Xtr en memoria:', 'Xtr' in dir())
Path('/content/oye_mia/out').mkdir(parents=True, exist_ok=True)
print('out creado')

data.npz existe: False
Xtr en memoria: True
out creado


In [ ]:
import torch, numpy as np, os
import torch.nn as nn
from pathlib import Path

class FCNBlock(nn.Module):
    def __init__(self, layer_dim):
        super().__init__()
        self.fcn_layer = nn.Linear(layer_dim, layer_dim)
        self.relu = nn.ReLU()
        self.layer_norm = nn.LayerNorm(layer_dim)
    def forward(self, x):
        return self.relu(self.layer_norm(self.fcn_layer(x)))

class Net(nn.Module):
    def __init__(self, input_shape=(16, 96), layer_dim=128, n_blocks=1, n_classes=1):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(input_shape[0] * input_shape[1], layer_dim)
        self.relu1 = nn.ReLU()
        self.layernorm1 = nn.LayerNorm(layer_dim)
        self.blocks = nn.ModuleList([FCNBlock(layer_dim) for _ in range(n_blocks)])
        self.last_layer = nn.Linear(layer_dim, n_classes)
        self.last_act = nn.Sigmoid()
    def forward(self, x):
        x = self.relu1(self.layernorm1(self.layer1(self.flatten(x))))
        for block in self.blocks:
            x = block(x)
        return self.last_act(self.last_layer(x))

OUT = Path('/content/oye_mia/out')
OUT.mkdir(parents=True, exist_ok=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
net = Net().to(dev)
opt = torch.optim.Adam(net.parameters(), lr=5e-4)
Xtr_t = torch.tensor(Xtr); ytr_t = torch.tensor(ytr)[:, None]
Xte_t = torch.tensor(Xte, device=dev); yte_t = torch.tensor(yte, device=dev)[:, None]
STEPS = 30000
best_rec = 0.0; best_state = None; best_step = -1
for step in range(1, STEPS + 1):
    net.train()
    idx = torch.randint(0, len(Xtr_t), (512,))
    xb, yb = Xtr_t[idx].to(dev), ytr_t[idx].to(dev)
    neg_w = 1 + 29 * (step / STEPS)
    w = torch.where(yb == 0, torch.tensor(neg_w, device=dev), torch.tensor(1.0, device=dev))
    pred = net(xb)
    loss = (torch.nn.functional.binary_cross_entropy(pred, yb, reduction='none') * w).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        net.eval()
        with torch.no_grad():
            pv = net(Xte_t)
        rec = ((pv > 0.5) & (yte_t == 1)).sum().item() / max(1, (yte_t == 1).sum().item())
        fp = int(((pv > 0.5) & (yte_t == 0)).sum().item())
        if step % 2000 == 0:
            print(f'paso {step} loss {loss.item():.4f} recall {rec:.3f} FP {fp}')
        if fp <= 5 and rec > best_rec:
            best_rec = rec; best_step = step
            best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
            print(f'  >> nuevo mejor en paso {step}: recall {rec:.3f} FP {fp} (en memoria)')
print('MEJOR:', best_rec, 'en paso', best_step)
assert best_state is not None, 'sin checkpoint valido'
torch.save(best_state, OUT / 'best.pt')
print('best.pt guardado:', (OUT / 'best.pt').exists(), os.path.getsize(OUT / 'best.pt'))
import onnx
net2 = Net(); net2.load_state_dict(best_state); net2 = net2.cpu().eval()
out_path = str(OUT / 'oye_mia.onnx')
torch.onnx.export(net2, torch.rand(1, 16, 96), out_path, opset_version=13, input_names=['input'], output_names=['output'], dynamic_axes=None, dynamo=False)
chk = onnx.load(out_path); onnx.checker.check_model(chk)
def shp(t):
    return [d.dim_value for d in t.type.tensor_type.shape.dim]
print('opset:', chk.opset_import[0].version)
print('entrada:', shp(chk.graph.input[0]), '| salida:', shp(chk.graph.output[0]))
print('tamano onnx KB:', os.path.getsize(out_path) / 1024)
from google.colab import files
files.download(out_path)
print('DESCARGA DISPARADA')

paso 2000 loss 0.0009 recall 0.880 FP 53
paso 4000 loss 0.0001 recall 0.872 FP 47
paso 6000 loss 0.0000 recall 0.868 FP 44
paso 8000 loss 0.0000 recall 0.868 FP 44
paso 10000 loss 0.0000 recall 0.870 FP 44
paso 12000 loss 0.0000 recall 0.874 FP 46
paso 14000 loss 0.0000 recall 0.876 FP 47
paso 16000 loss 0.0000 recall 0.878 FP 46
paso 18000 loss 0.0000 recall 0.876 FP 47
paso 20000 loss 0.0000 recall 0.874 FP 47
paso 22000 loss 0.0000 recall 0.860 FP 44
paso 24000 loss 0.0000 recall 0.858 FP 41
paso 26000 loss 0.0000 recall 0.856 FP 41
paso 28000 loss 0.0000 recall 0.852 FP 41
paso 30000 loss 0.0000 recall 0.854 FP 41
MEJOR: 0.0 en paso -1


AssertionError: sin checkpoint valido

In [ ]:
import torch, numpy as np, os
import torch.nn as nn
from pathlib import Path

class FCNBlock(nn.Module):
    def __init__(self, layer_dim):
        super().__init__()
        self.fcn_layer = nn.Linear(layer_dim, layer_dim)
        self.relu = nn.ReLU()
        self.layer_norm = nn.LayerNorm(layer_dim)
    def forward(self, x):
        return self.relu(self.layer_norm(self.fcn_layer(x)))

class Net(nn.Module):
    def __init__(self, input_shape=(16, 96), layer_dim=128, n_blocks=1, n_classes=1):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(input_shape[0] * input_shape[1], layer_dim)
        self.relu1 = nn.ReLU()
        self.layernorm1 = nn.LayerNorm(layer_dim)
        self.blocks = nn.ModuleList([FCNBlock(layer_dim) for _ in range(n_blocks)])
        self.last_layer = nn.Linear(layer_dim, n_classes)
        self.last_act = nn.Sigmoid()
    def forward(self, x):
        x = self.relu1(self.layernorm1(self.layer1(self.flatten(x))))
        for block in self.blocks:
            x = block(x)
        return self.last_act(self.last_layer(x))

OUT = Path('/content/oye_mia/out')
OUT.mkdir(parents=True, exist_ok=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(7)
net = Net().to(dev)
opt = torch.optim.Adam(net.parameters(), lr=5e-4)
Xtr_t = torch.tensor(Xtr); ytr_t = torch.tensor(ytr)[:, None]
Xte_t = torch.tensor(Xte, device=dev); yte_t = torch.tensor(yte, device=dev)[:, None]
npos = max(1, (yte_t == 1).sum().item()); nneg = max(1, (yte_t == 0).sum().item())
STEPS = 30000
best = None; fallback = None
for step in range(1, STEPS + 1):
    net.train()
    idx = torch.randint(0, len(Xtr_t), (512,))
    xb, yb = Xtr_t[idx].to(dev), ytr_t[idx].to(dev)
    neg_w = 1 + 29 * (step / STEPS)
    w = torch.where(yb == 0, torch.tensor(neg_w, device=dev), torch.tensor(1.0, device=dev))
    pred = net(xb)
    loss = (torch.nn.functional.binary_cross_entropy(pred, yb, reduction='none') * w).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        net.eval()
        with torch.no_grad():
            pv = net(Xte_t)
        for th in (0.5, 0.6, 0.7, 0.8):
            rec = ((pv > th) & (yte_t == 1)).sum().item() / npos
            fp = int(((pv > th) & (yte_t == 0)).sum().item())
            if fp <= 10 and (best is None or rec > best[0]):
                best = (rec, fp, th, step, {k: v.cpu().clone() for k, v in net.state_dict().items()})
                print(f'  >> mejor: paso {step} umbral {th} recall {rec:.3f} FP {fp}')
            score = rec - 2.0 * (fp / nneg)
            if fallback is None or score > fallback[0]:
                fallback = (score, rec, fp, th, step, {k: v.cpu().clone() for k, v in net.state_dict().items()})
        if step % 2000 == 0:
            rec5 = ((pv > 0.5) & (yte_t == 1)).sum().item() / npos
            fp5 = int(((pv > 0.5) & (yte_t == 0)).sum().item())
            print(f'paso {step} loss {loss.item():.4f} recall@0.5 {rec5:.3f} FP@0.5 {fp5}')
if best is not None:
    rec, fp, th, step, state = best
    print(f'ELEGIDO: paso {step} umbral {th} recall {rec:.3f} FP {fp}')
else:
    score, rec, fp, th, step, state = fallback
    print(f'SIN checkpoint con FP<=10; respaldo: paso {step} umbral {th} recall {rec:.3f} FP {fp}')
torch.save(state, OUT / 'best.pt')
print('best.pt guardado:', (OUT / 'best.pt').exists(), os.path.getsize(OUT / 'best.pt'))
import onnx
net2 = Net(); net2.load_state_dict(state); net2 = net2.cpu().eval()
out_path = str(OUT / 'oye_mia.onnx')
torch.onnx.export(net2, torch.rand(1, 16, 96), out_path, opset_version=13, input_names=['input'], output_names=['output'], dynamic_axes=None, dynamo=False)
chk = onnx.load(out_path); onnx.checker.check_model(chk)
def shp(t):
    return [d.dim_value for d in t.type.tensor_type.shape.dim]
print('opset:', chk.opset_import[0].version)
print('entrada:', shp(chk.graph.input[0]), '| salida:', shp(chk.graph.output[0]))
print('tamano onnx KB:', os.path.getsize(out_path) / 1024)
from google.colab import files
files.download(out_path)
print('DESCARGA DISPARADA')

paso 2000 loss 0.0442 recall@0.5 0.856 FP@0.5 52
paso 4000 loss 0.0001 recall@0.5 0.868 FP@0.5 48
paso 6000 loss 0.0000 recall@0.5 0.864 FP@0.5 47
paso 8000 loss 0.0000 recall@0.5 0.864 FP@0.5 47
paso 10000 loss 0.0000 recall@0.5 0.866 FP@0.5 47
paso 12000 loss 0.0000 recall@0.5 0.868 FP@0.5 50
paso 14000 loss 0.0000 recall@0.5 0.868 FP@0.5 52
paso 16000 loss 0.0000 recall@0.5 0.868 FP@0.5 52
paso 18000 loss 0.0000 recall@0.5 0.868 FP@0.5 52
paso 20000 loss 0.0000 recall@0.5 0.868 FP@0.5 54
paso 22000 loss 0.0000 recall@0.5 0.864 FP@0.5 50
paso 24000 loss 0.0000 recall@0.5 0.862 FP@0.5 48
paso 26000 loss 0.0000 recall@0.5 0.862 FP@0.5 45
paso 28000 loss 0.0000 recall@0.5 0.864 FP@0.5 45
paso 30000 loss 0.0000 recall@0.5 0.864 FP@0.5 46
SIN checkpoint con FP<=10; respaldo: paso 500 umbral 0.7 recall 0.822 FP 30
best.pt guardado: True 858901
opset: 13
entrada: [1, 16, 96] | salida: [1, 1]
tamano onnx KB: 839.2412109375


/tmp/ipykernel_1223/1640715369.py:78: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(net2, torch.rand(1, 16, 96), out_path, opset_version=13, input_names=['input'], output_names=['output'], dynamic_axes=None, dynamo=False)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DESCARGA DISPARADA


In [ ]:
import torch, numpy as np, os
import torch.nn as nn
from pathlib import Path

class FCNBlock(nn.Module):
    def __init__(self, layer_dim):
        super().__init__()
        self.fcn_layer = nn.Linear(layer_dim, layer_dim)
        self.relu = nn.ReLU()
        self.layer_norm = nn.LayerNorm(layer_dim)
    def forward(self, x):
        return self.relu(self.layer_norm(self.fcn_layer(x)))

class Net(nn.Module):
    def __init__(self, input_shape=(16, 96), layer_dim=128, n_blocks=1, n_classes=1):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(input_shape[0] * input_shape[1], layer_dim)
        self.relu1 = nn.ReLU()
        self.layernorm1 = nn.LayerNorm(layer_dim)
        self.blocks = nn.ModuleList([FCNBlock(layer_dim) for _ in range(n_blocks)])
        self.last_layer = nn.Linear(layer_dim, n_classes)
        self.last_act = nn.Sigmoid()
    def forward(self, x):
        x = self.relu1(self.layernorm1(self.layer1(self.flatten(x))))
        for block in self.blocks:
            x = block(x)
        return self.last_act(self.last_layer(x))

OUT = Path('/content/oye_mia/out')
OUT.mkdir(parents=True, exist_ok=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(123)
net = Net().to(dev)
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
Xtr_t = torch.tensor(Xtr); ytr_t = torch.tensor(ytr)[:, None]
Xte_t = torch.tensor(Xte, device=dev); yte_t = torch.tensor(yte, device=dev)[:, None]
npos = max(1, (yte_t == 1).sum().item()); nneg = max(1, (yte_t == 0).sum().item())
STEPS = 30000
best = None; fallback = None
for step in range(1, STEPS + 1):
    net.train()
    idx = torch.randint(0, len(Xtr_t), (512,))
    xb, yb = Xtr_t[idx].to(dev), ytr_t[idx].to(dev)
    neg_w = 1 + 299 * (step / STEPS)
    w = torch.where(yb == 0, torch.tensor(neg_w, device=dev), torch.tensor(1.0, device=dev))
    pred = net(xb)
    loss = (torch.nn.functional.binary_cross_entropy(pred, yb, reduction='none') * w).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        net.eval()
        with torch.no_grad():
            pv = net(Xte_t)
        for th in (0.5, 0.6, 0.7, 0.8):
            rec = ((pv > th) & (yte_t == 1)).sum().item() / npos
            fp = int(((pv > th) & (yte_t == 0)).sum().item())
            if fp <= 10 and (best is None or rec > best[0]):
                best = (rec, fp, th, step, {k: v.cpu().clone() for k, v in net.state_dict().items()})
                print(f'  >> mejor: paso {step} umbral {th} recall {rec:.3f} FP {fp}')
            score = rec - 2.0 * (fp / nneg)
            if fallback is None or score > fallback[0]:
                fallback = (score, rec, fp, th, step, {k: v.cpu().clone() for k, v in net.state_dict().items()})
        if step % 2000 == 0:
            rec5 = ((pv > 0.5) & (yte_t == 1)).sum().item() / npos
            fp5 = int(((pv > 0.5) & (yte_t == 0)).sum().item())
            print(f'paso {step} loss {loss.item():.4f} recall@0.5 {rec5:.3f} FP@0.5 {fp5}')
if best is not None:
    rec, fp, th, step, state = best
    print(f'ELEGIDO: paso {step} umbral {th} recall {rec:.3f} FP {fp}')
else:
    score, rec, fp, th, step, state = fallback
    print(f'SIN checkpoint con FP<=10; respaldo: paso {step} umbral {th} recall {rec:.3f} FP {fp}')
torch.save(state, OUT / 'best.pt')
print('best.pt guardado:', (OUT / 'best.pt').exists(), os.path.getsize(OUT / 'best.pt'))
import onnx
net2 = Net(); net2.load_state_dict(state); net2 = net2.cpu().eval()
out_path = str(OUT / 'oye_mia.onnx')
torch.onnx.export(net2, torch.rand(1, 16, 96), out_path, opset_version=13, input_names=['input'], output_names=['output'], dynamic_axes=None, dynamo=False)
chk = onnx.load(out_path); onnx.checker.check_model(chk)
def shp(t):
    return [d.dim_value for d in t.type.tensor_type.shape.dim]
print('opset:', chk.opset_import[0].version)
print('entrada:', shp(chk.graph.input[0]), '| salida:', shp(chk.graph.output[0]))
print('tamano onnx KB:', os.path.getsize(out_path) / 1024)
from google.colab import files
files.download(out_path)
print('DESCARGA DISPARADA')

paso 2000 loss 0.0630 recall@0.5 0.840 FP@0.5 50
paso 4000 loss 0.1488 recall@0.5 0.790 FP@0.5 36
paso 6000 loss 0.0015 recall@0.5 0.860 FP@0.5 52
  >> mejor: paso 6500 umbral 0.5 recall 0.632 FP 9
paso 8000 loss 0.0017 recall@0.5 0.860 FP@0.5 48
paso 10000 loss 0.0993 recall@0.5 0.814 FP@0.5 36
paso 12000 loss 0.0006 recall@0.5 0.850 FP@0.5 44
paso 14000 loss 1.1709 recall@0.5 0.398 FP@0.5 7
paso 16000 loss 0.0585 recall@0.5 0.854 FP@0.5 57
paso 18000 loss 0.0001 recall@0.5 0.870 FP@0.5 68
paso 20000 loss 0.0000 recall@0.5 0.872 FP@0.5 66
paso 22000 loss 0.0000 recall@0.5 0.872 FP@0.5 63
paso 24000 loss 0.0000 recall@0.5 0.872 FP@0.5 62
paso 26000 loss 0.0000 recall@0.5 0.874 FP@0.5 62
paso 28000 loss 0.0000 recall@0.5 0.874 FP@0.5 61
paso 30000 loss 0.0000 recall@0.5 0.872 FP@0.5 61
ELEGIDO: paso 6500 umbral 0.5 recall 0.632 FP 9
best.pt guardado: True 858901
opset: 13
entrada: [1, 16, 96] | salida: [1, 1]
tamano onnx KB: 839.2412109375


/tmp/ipykernel_1223/602007850.py:78: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(net2, torch.rand(1, 16, 96), out_path, opset_version=13, input_names=['input'], output_names=['output'], dynamic_axes=None, dynamo=False)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DESCARGA DISPARADA
